# Kata 1 — Agentic Loop & Deterministic Control

**Claude Architecture Certification — reference notebook**

This notebook is the canonical reference for Kata 1 of the `ia-claude` workshop. It walks through every Claude / Anthropic concept the kata exercises, the architecture, the patterns, and the constitutional principles enforced.

It is **additive** to `README.md`: the README stays the in-repo summary; this notebook is the certification-grade deep dive.

> **TL;DR.** Build an agent loop whose only termination signal is the structured `stop_reason` field returned by the Anthropic Messages API. Never inspect prose. Every iteration writes one line to a JSONL audit log. Two runs against the same fixture produce byte-identical `(stop_signal, branch_taken)` columns.

## Section 1 — Concepts (Claude architecture certification)

Every concept this kata exercises, with a one-line definition tied to the certification syllabus and the file that demonstrates it.

| Concept | One-line definition | Where it lives in this kata |
|---|---|---|
| **Agentic loop** | A control structure that calls the model, inspects the response, dispatches tools, and re-calls the model until a terminal signal fires. | `loop.py::run` |
| **Signal-driven control via `stop_reason`** | Branch the loop on the structured `stop_reason` field (`tool_use`, `end_turn`, `max_tokens`, `stop_sequence`) — never on the model's prose. | `loop.py::classify_stop_signal` + the five branches in `run` |
| **Tool use round-trip (`tool_use` ↔ `tool_result`)** | Model emits a `tool_use` block; the loop runs the tool and appends a `tool_result` block to history; the model continues with that result in context. | `loop.py` tool_use branch + `tools.py::ToolRegistry.dispatch` |
| **Structured output / schema-enforced boundaries** | Every payload crossing a module boundary is validated against a declared schema (pydantic v2 + JSON Schema). | `models.py`, `contracts/*.schema.json` |
| **Event-log provenance** | An append-only JSONL audit trail captures one record per loop iteration, sufficient to reconstruct the full trajectory offline. | `events.py::EventLog` |
| **Deterministic replay** | Two runs against the same recorded session produce byte-identical `(stop_signal, branch_taken)` sequences. | `events.set_frozen_clock` + `serialize_record` |
| **Fixture-vs-live SDK seam** | A thin injectable client lets tests swap a recorded-response client for the real SDK without changing loop logic. | `client.py::RecordedClient` / `LiveClient` |
| **Hash-locked acceptance criteria** | `.feature` files are produced by `/iikit-04-testify` and locked via `.specify/context.json` hashes; tests fix code, never assertions. | `tests/katas/kata_001_agentic_loop/features/*.feature` |
| **Pre-commit assertion-integrity hook** | Git pre-commit hook verifies the locked test set hashes still match — blocks commits that mutate assertions. | `.git/hooks/pre-commit` |
| **MalformedToolUse halt (FR-008)** | When the model names an unregistered tool or sends a payload that fails the tool's `input_schema`, the loop halts with a labelled cause — no heuristic recovery. | `tools.py::MalformedToolUse` + `loop.py` catch site |
| **Structured tool-error recovery (FR-007)** | A tool raising an exception is mechanically wrapped as `ToolResult(status='error', error_category=...)`; the loop continues under signal-driven rules. | `tools.py::ToolRegistry.dispatch` |

## Section 2 — Architecture walkthrough

Components and the data flow between them. The CLI drives a session; the **Agentic Loop** is the *single decision point* and branches solely on `stop_reason`; the **Tool Registry** holds declared tool schemas and is consulted synchronously per turn; the **Event Log** captures every iteration for replay; the SDK seam lets recorded fixtures stand in for the live API.

```mermaid
flowchart LR
    CLI[Practitioner CLI<br/>runner.py] --> RS[RuntimeSession<br/>session.py]
    RS --> Loop[Agentic Loop<br/>loop.run]
    Loop -- send --> Client[Client seam<br/>client.py]
    Client -- fixtures --> Rec[RecordedClient]
    Client -- live API --> Live[LiveClient]
    Loop -- dispatch --> Reg[ToolRegistry<br/>tools.py]
    Loop -- emit EventRecord --> Log[(events.jsonl<br/>events.py)]
    Log -- reconstruct_trajectory --> Replay[replay.py]
```

ASCII fallback (for non-mermaid renderers):

```text
┌────────────────────────────┐
│ Practitioner CLI (runner)  │
└──────────────┬─────────────┘
               │
┌──────────────┴─────────────┐      ┌──────────────────────┐
│      Agentic Loop          │ ───▶ │     Tool Registry    │
│ (loop.run / branch on      │      └──────────────────────┘
│  stop_signal only)         │
└──────────────┬─────────────┘
               │ emit(EventRecord)
┌──────────────┴─────────────┐      ┌──────────────────────┐
│ Event Log (JSONL, append)  │ ◀─── │ RecordedClient       │ (offline tests)
└────────────────────────────┘      │ LiveClient           │ (real Messages API)
                                    └──────────────────────┘
```

### Module map

| Module | Role | Why this seam exists |
|---|---|---|
| `loop.py` | Single decision point. Five labelled branches (`end_turn`, `tool_use`, `max_tokens`, `stop_sequence`, unhandled/absent). | Centralising the switch makes Principle I auditable and AST-lintable. |
| `client.py` | `RawResponse` dataclass + `RecordedClient` + `LiveClient`. | Inject the SDK so tests stay deterministic; loop logic never sees SDK objects. |
| `tools.py` | `ToolRegistry`, `ToolImpl`, `MalformedToolUse`. | Pre-dispatch validation halts on FR-008; routine errors recover via FR-007 — both *without* reading text. |
| `events.py` | `EventLog`, `serialize_record`, `set_frozen_clock`. | Append-only JSONL with `extra="forbid"` so prose-derived columns are structurally impossible. |
| `models.py` | Pydantic v2 boundary models. | One source of truth for every cross-module shape. |
| `session.py` | `RuntimeSession` (history + registry + log). | Keeps `loop.run` a pure function over its inputs — required for SC-007 / SC-008 reproducibility. |
| `replay.py` | `reconstruct_trajectory(events_path)`. | Proves SC-008: log alone is sufficient to recover trajectory. |
| `runner.py` | CLI: `python -m katas.kata_001_agentic_loop.runner`. | Quickstart-grade entrypoint — fixture or `LIVE_API=1`. |

## Section 3 — Patterns and the trade-off each one solves

| Pattern | What it does | Trade-off it solves |
|---|---|---|
| **Signal-over-prose** | Branch only on structured metadata (`stop_reason`). | Prose-matching breaks the moment the model phrases "done" differently; signal-driven code is observable and reproducible. (Constitution Principle I.) |
| **Schema-first contract** | Tool calls / results / event records are pydantic v2 models with `extra="forbid"`. | Drift between modules is caught at construction; absent schemas push errors to production. (Principle II.) |
| **Pure-function transition** | `loop.run` reads from / writes to a `RuntimeSession`; transitions are explicit and labelled. | Easy to unit-test every branch in isolation and to replay deterministically. |
| **Append-only event log** | One JSONL line per iteration, no in-place mutation. | Audit trail survives crashes; supports cheap diff-based reproducibility checks. (Principle VII.) |
| **Replay-from-log determinism** | `reconstruct_trajectory` rebuilds (iterations, tool_invocations, termination_cause) from the JSONL alone. | Decouples reviewers from in-memory state; an outsider can audit a run without re-executing the model. (SC-008.) |
| **Injectable SDK seam** | `RecordedClient` / `LiveClient` share the `send()` surface. | Tests run offline + deterministic; live mode is one env-var flip away. |
| **AST lint as structural guard** | `tests/.../lint/test_no_prose_matching.py` walks the AST of `loop.py`. | Prose matching cannot be re-introduced by accident — the build fails before review. |

## Section 4 — Principles enforced + Anthropic engineering recommendations

Constitutional principles touched by this kata, cross-referenced with Anthropic's published agent-engineering recommendations, and a practitioner checklist for applying signal-driven loops to a real agent.

### Constitution → file-level enforcement

| Principle | What it requires | Where it is enforced |
|---|---|---|
| **I. Determinism Over Probability** *(NN)* | Loop branches solely on structured signals. | `loop.py` switch + `tests/.../lint/test_no_prose_matching.py` AST lint + `unit/test_decoy_phrases.py` |
| **II. Schema-Enforced Boundaries** *(NN)* | Every payload validated on construction. | `models.py` + `contracts/*.schema.json` + `unit/test_event_log_shape.py` |
| **V. Test-First Kata Delivery** *(NN)* | `.feature` files exist + fail before code is written. | Generated by `/iikit-04-testify`, hash-locked in `.specify/context.json`. |
| **VII. Provenance & Self-Audit** | Append-only audit trail sufficient to reproduce the run. | `events.py` + `replay.py` + `unit/test_trajectory_reconstruction.py` |
| **VIII. Mandatory Documentation** *(NN)* | Why-comments + per-kata README. | Module + function docstrings, `README.md`, this notebook. |

### Anthropic engineering recommendations exercised

- **Use the Messages API's `stop_reason`, not text inspection, to drive loops.** ([Tool use docs](https://docs.anthropic.com/claude/docs/tool-use).) The kata is the canonical demonstration.
- **Validate tool inputs with declared JSON schemas before dispatch.** Implemented by `ToolRegistry.validate_call` using `jsonschema.Draft202012Validator`.
- **Convert tool exceptions into structured `tool_result` blocks with `is_error=True`** so the model can recover. Implemented in `loop.run` tool_use branch.
- **Make agent runs auditable.** One JSONL record per iteration; the trajectory is reconstructible without replay.
- **Keep the SDK behind a seam in tests.** `RecordedClient` plays back recorded turns deterministically.

### Practitioner checklist — applying signal-driven loops to a real agent

1. **List every termination cause you support** *before* writing the loop. If you don't have a labelled bucket for `max_tokens`, you'll silently drop those runs.
2. **Map the closed set of structured signals.** Use `Literal` / enums; route everything outside that set into a single `unhandled_signal` halt.
3. **Forbid prose inspection in the loop module by lint.** AST checks are cheap and catch the regression before review.
4. **Schema-enforce both directions of the tool round-trip.** Inputs validated against `input_schema`; outputs serialised to a structured `tool_result` block.
5. **Catch tool exceptions broadly inside the dispatcher** and re-emit them as `ToolResult(status='error')`. Do not let the loop see Python exceptions from tools.
6. **Log one record per iteration**, append-only, schema-validated. Make the log the ground truth — never the in-memory state.
7. **Provide a frozen-clock test mode** so reproducibility diffs are byte-identical.
8. **Inject the SDK** behind a `send(messages, tools) -> RawResponse` interface so tests can run offline.

## Section 5 — Live walkthrough (executable cells)

The remaining cells exercise the kata end-to-end against the recorded `happy_path` fixture, then prove the SC-007 byte-identical-diff property and the SC-008 reconstruct-from-log property — all without hitting the live API.

In [ ]:
# Boilerplate: make the kata package importable from this notebook regardless
# of where Jupyter is launched from. We resolve the repo root by walking up
# until we find ``pyproject.toml``.
from __future__ import annotations

import sys
from pathlib import Path

here = Path.cwd()
while not (here / "pyproject.toml").exists() and here != here.parent:
    here = here.parent
REPO_ROOT = here
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("repo root:", REPO_ROOT)

In [ ]:
# Build a session against the recorded happy_path fixture (tool_use → end_turn).
import uuid
from datetime import datetime, timezone

from katas.kata_001_agentic_loop.client import RecordedClient
from katas.kata_001_agentic_loop.events import set_frozen_clock
from katas.kata_001_agentic_loop.loop import run
from katas.kata_001_agentic_loop.models import ToolDefinition
from katas.kata_001_agentic_loop.session import RuntimeSession

FIXTURE_ROOT = REPO_ROOT / "tests" / "katas" / "kata_001_agentic_loop" / "fixtures"
RUNS_ROOT = REPO_ROOT / "runs" / "notebook"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

weather_def = ToolDefinition(
    name="get_weather",
    description="Stub weather tool.",
    input_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
        "additionalProperties": False,
    },
)
weather_impl = lambda payload: {"city": payload["city"], "temp_c": 18, "conditions": "clear"}

set_frozen_clock(datetime(2026, 1, 1, 0, 0, 0, tzinfo=timezone.utc))

session = RuntimeSession.open(
    session_id=str(uuid.uuid4()),
    model="claude-test",
    client=RecordedClient.from_fixture(FIXTURE_ROOT / "happy_path.json"),
    runs_root=RUNS_ROOT,
    tool_definitions=[(weather_def, weather_impl)],
)
termination = run(session, "What's the weather in Bogotá?")
session.close()
print("termination cause:", termination)
print("events log:", session.event_log.path)

In [ ]:
# Inspect the JSONL event log line-by-line. One record per iteration.
import json

for line in session.event_log.path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        record = json.loads(line)
        print(json.dumps(record, indent=2, sort_keys=True))

In [ ]:
# SC-008 — reconstruct trajectory from the event log alone. No model text, no in-memory state.
from katas.kata_001_agentic_loop.replay import reconstruct_trajectory

summary = reconstruct_trajectory(session.event_log.path)
print(summary)
assert summary.iterations == 2
assert summary.tool_invocations == 1
assert summary.termination_cause == "end_turn"
print("SC-008 OK — log alone reconstructs (iterations=2, tool_invocations=1, termination='end_turn')")

In [ ]:
# SC-007 — two independent runs against the same fixture produce byte-identical (stop_signal, branch_taken) columns.
def cols(path):
    return [
        (json.loads(line)["stop_signal"], json.loads(line)["branch_taken"])
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

session_b = RuntimeSession.open(
    session_id=str(uuid.uuid4()),
    model="claude-test",
    client=RecordedClient.from_fixture(FIXTURE_ROOT / "happy_path.json"),
    runs_root=RUNS_ROOT,
    tool_definitions=[(weather_def, weather_impl)],
)
run(session_b, "What's the weather in Bogotá?")
session_b.close()

a_cols = cols(session.event_log.path)
b_cols = cols(session_b.event_log.path)
print("run A columns:", a_cols)
print("run B columns:", b_cols)
assert a_cols == b_cols
print("SC-007 OK — (stop_signal, branch_taken) sequences are byte-identical across two runs.")

In [ ]:
# Anti-pattern guard: feed the loop a turn whose prose says 'task complete'
# but whose structured signal is still tool_use — assert the loop continues.
decoy_session = RuntimeSession.open(
    session_id=str(uuid.uuid4()),
    model="claude-test",
    client=RecordedClient.from_fixture(FIXTURE_ROOT / "decoy_phrase.json"),
    runs_root=RUNS_ROOT,
    tool_definitions=[(weather_def, weather_impl)],
)
decoy_cause = run(decoy_session, "x")
decoy_session.close()
print("decoy run termination:", decoy_cause)
for line in decoy_session.event_log.path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        print(json.loads(line))
assert decoy_cause == "end_turn", "prose 'task complete' must NOT trigger termination"
print("Anti-pattern OK — decoy phrases were ignored; only end_turn signal terminated the loop.")

In [ ]:
# Malformed tool_use halt (FR-008) — model names an unregistered tool.
malformed_session = RuntimeSession.open(
    session_id=str(uuid.uuid4()),
    model="claude-test",
    client=RecordedClient.from_fixture(FIXTURE_ROOT / "malformed_tool_use.json"),
    runs_root=RUNS_ROOT,
    tool_definitions=[(weather_def, weather_impl)],
)
malformed_cause = run(malformed_session, "x")
malformed_session.close()
print("malformed-tool-use cause:", malformed_cause)
assert malformed_cause == "malformed_tool_use"
print("FR-008 OK — labelled halt instead of heuristic recovery.")

In [ ]:
# Tool error recovery (FR-007) — tool implementation raises; loop continues.
def boom(_payload):
    raise RuntimeError("tool blew up")

boom_def = ToolDefinition(
    name="boom",
    description="A tool that always raises.",
    input_schema={"type": "object", "properties": {"detonator": {"type": "boolean"}}},
)
tool_err_session = RuntimeSession.open(
    session_id=str(uuid.uuid4()),
    model="claude-test",
    client=RecordedClient.from_fixture(FIXTURE_ROOT / "tool_error.json"),
    runs_root=RUNS_ROOT,
    tool_definitions=[(boom_def, boom)],
)
tool_err_cause = run(tool_err_session, "x")
tool_err_session.close()
print("tool-error termination:", tool_err_cause)
for line in tool_err_session.event_log.path.read_text(encoding="utf-8").splitlines():
    if line.strip():
        print(json.loads(line))
assert tool_err_cause == "end_turn", "tool errors must NOT halt the loop"
print("FR-007 OK — tool exception captured as ToolResult(error); loop continued to genuine end_turn.")

## Section 6 — Result summary

What the live walkthrough above proved end-to-end:

| Property | Demonstrated by | Outcome |
|---|---|---|
| **Signal-driven termination (US1)** | `happy_path` run → `termination='end_turn'` | ✅ |
| **SC-008 — reconstruct from log alone** | `replay.reconstruct_trajectory` | ✅ `(iterations=2, tool_invocations=1, cause='end_turn')` |
| **SC-007 — byte-identical diff across runs** | Two runs against `happy_path` | ✅ identical `(stop_signal, branch_taken)` |
| **Decoy-phrase defence (US2)** | `decoy_phrase` fixture with prose 'task complete' but `tool_use` signal | ✅ loop continued; halted on the *next* genuine `end_turn` |
| **Malformed tool_use halt (FR-008)** | `malformed_tool_use` fixture | ✅ labelled halt `termination='malformed_tool_use'` |
| **Tool-error recovery (FR-007)** | `tool_error` fixture; tool raises | ✅ structured `ToolResult(error)`; loop continued |

And from the test suite (run separately):

```
46 tests pass — BDD US1×6 + US2×7 + US3×4, AST lint×4, unit×27, runner×2, history-replay×2
loop.py line coverage: 97% (target ≥ 90%)
Total kata-package coverage: 91%
ruff check + ruff format --check: clean
```

## Section 7 — Where to go next

- Read `katas/kata_001_agentic_loop/README.md` for the in-repo summary.
- Read `specs/001-agentic-loop/spec.md` and `plan.md` for the requirements/design split.
- Read `specs/001-agentic-loop/contracts/event-log-record.schema.json` to see what the auditor will see.
- Read `tests/katas/kata_001_agentic_loop/lint/test_no_prose_matching.py` to see exactly which AST patterns are forbidden.
- The next kata in the workshop builds on this loop — keep it a pure, signal-driven function and the rest of the syllabus stays tractable.